In [ ]:
#Libraries installation
!pip install -U scikit-learn
!pip install librosa matplotlib umap-learn

Uploading Dataset


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

file_path = '/content/drive/My Drive/cse425/dataset_final.csv'
data = pd.read_csv(file_path)

mfcc_columns = [f'mfcc_{i}' for i in range(1, 14)]
mfcc_features = data[mfcc_columns]

tfidf_columns = [col for col in data.columns if col.startswith('tfidf_')]
tfidf_features = data[tfidf_columns]

combined_features = pd.concat([mfcc_features, tfidf_features], axis=1)

scaler = StandardScaler()
combined_features_scaled = scaler.fit_transform(combined_features)

print(f"Shape of scaled combined features: {combined_features_scaled.shape}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import DBSCAN
from sklearn.metrics import davies_bouldin_score
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from sklearn.manifold import TSNEimport pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

correlation heatmap

In [ ]:
mfcc_columns = [f'mfcc_{i}' for i in range(1, 14)]
mfcc_features = data[mfcc_columns]

tfidf_columns = [col for col in data.columns if col.startswith('tfidf_')]
tfidf_features = data[tfidf_columns]

combined_features = pd.concat([mfcc_features, tfidf_features], axis=1)
correlation_matrix = combined_features.corr()

plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of MFCC and TF-IDF Features')
plt.show()

In [ ]:
latent_dim = 2

inputs = layers.Input(shape=(combined_features_scaled.shape[1],))
x = layers.Reshape((combined_features_scaled.shape[1], 1))(inputs)
x = layers.Conv1D(32, kernel_size=3, activation='relu', padding='same')(x)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.Conv1D(64, kernel_size=3, activation='relu', padding='same')(x)
x = layers.MaxPooling1D(pool_size=2)(x)
x = layers.Flatten()(x)
z_mean_encoder = layers.Dense(latent_dim, name='z_mean')(x)
z_log_var_encoder = layers.Dense(latent_dim, name='z_log_var')(x)

encoder = models.Model(inputs, [z_mean_encoder, z_log_var_encoder], name='encoder')

class Sampling(layers.Layer):
    """Uses (z_mean, z_log_var) to sample z, the vector encoding a digit."""
    def call(self, inputs):
        z_mean, z_log_var = inputs
        batch = tf.shape(z_mean)[0]
        dim = tf.shape(z_mean)[1]
        epsilon = tf.keras.backend.random_normal(shape=(batch, dim))
        return z_mean + tf.exp(0.5 * z_log_var) * epsilon

latent_inputs = layers.Input(shape=(latent_dim,))
h_decoded = layers.Dense(64, activation='relu')(latent_inputs)
x_decoded_mean_decoder = layers.Dense(combined_features_scaled.shape[1], activation=None)(h_decoded)

decoder = models.Model(latent_inputs, x_decoded_mean_decoder, name='decoder')

class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.sampling = Sampling()

    def call(self, inputs):
        z_mean, z_log_var = self.encoder(inputs)
        z = self.sampling([z_mean, z_log_var])
        reconstruction = self.decoder(z)

        reconstruction_loss = tf.reduce_sum(tf.square(inputs - reconstruction), axis=1)

        kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
        kl_loss = tf.reduce_sum(kl_loss, axis=1)

        total_loss = tf.reduce_mean(reconstruction_loss + kl_loss)
        self.add_loss(total_loss)

        return reconstruction

vae = VAE(encoder, decoder)
vae.compile(optimizer='rmsprop')


vae.fit(combined_features_scaled, epochs=50, batch_size=64)

Latent Features

In [ ]:
z_mean_predicted, z_log_var_predicted = encoder.predict(combined_features_scaled, batch_size=64)

epsilon = tf.keras.backend.random_normal(shape=tf.shape(z_mean_predicted))
latent_features = z_mean_predicted + tf.exp(0.5 * z_log_var_predicted) * epsilon

import pandas as pd
latent_features_df = pd.DataFrame(latent_features, columns=['latent_dim_1', 'latent_dim_2'])
print(latent_features_df.head())

K means Silhouette Score

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(latent_features)

sil_score_kmeans = silhouette_score(latent_features, kmeans_labels)
print(f'Silhouette Score (K-Means): {sil_score_kmeans}')

Agglomerative Clustering Silhouette Score


In [ ]:
agg_clust = AgglomerativeClustering(n_clusters=3)
agg_labels = agg_clust.fit_predict(latent_features)

agg_sil_score = silhouette_score(latent_features, agg_labels)
print(f'Agglomerative Clustering Silhouette Score: {agg_sil_score}')

DBSCAN Silhouette Score

In [ ]:
dbscan = DBSCAN(eps=0.5, min_samples=5)
db_labels = dbscan.fit_predict(latent_features)

db_sil_score = silhouette_score(latent_features, db_labels)
print(f'DBSCAN Silhouette Score: {db_sil_score}')

Davies-Bouldin Index -- K-Means

In [ ]:
db_index_kmeans = davies_bouldin_score(latent_features, kmeans_labels)
print(f'Davies-Bouldin Index (K-Means): {db_index_kmeans}')

Adjusted Rand Index

In [ ]:
true_labels = data['genre'].values
ari_score = adjusted_rand_score(true_labels, kmeans_labels)
print(f'Adjusted Rand Index (ARI): {ari_score}')

In [ ]:
# K-Means:
sil_score_kmeans = silhouette_score(latent_features, kmeans_labels)
db_index_kmeans = davies_bouldin_score(latent_features, kmeans_labels)
ari_score_kmeans = adjusted_rand_score(true_labels, kmeans_labels) if 'true_labels' in locals() else None

print("K-Means Performance:")
print(f"Silhouette Score: {sil_score_kmeans}")
print(f"Davies-Bouldin Index: {db_index_kmeans}")
print(f"Adjusted Rand Index: {ari_score_kmeans if ari_score_kmeans is not None else 'N/A'}")

In [ ]:
# Agglomerative Clustering:
sil_score_agg = silhouette_score(latent_features, agg_labels)
db_index_agg = davies_bouldin_score(latent_features, agg_labels)
ari_score_agg = adjusted_rand_score(true_labels, agg_labels) if 'true_labels' in locals() else None

print("\nAgglomerative Clustering Performance:")
print(f"Silhouette Score: {sil_score_agg}")
print(f"Davies-Bouldin Index: {db_index_agg}")
print(f"Adjusted Rand Index: {ari_score_agg if ari_score_agg is not None else 'N/A'}")

In [ ]:
# DBSCAN:
sil_score_dbscan = silhouette_score(latent_features, db_labels)
db_index_dbscan = davies_bouldin_score(latent_features, db_labels)
ari_score_dbscan = adjusted_rand_score(true_labels, db_labels) if 'true_labels' in locals() else None

print("\nDBSCAN Performance:")
print(f"Silhouette Score: {sil_score_dbscan}")
print(f"Davies-Bouldin Index: {db_index_dbscan}")
print(f"Adjusted Rand Index: {ari_score_dbscan if ari_score_dbscan is not None else 'N/A'}")

In [ ]:
pca = PCA(n_components=2, random_state=42)
latent_2d_pca = pca.fit_transform(latent_features)

kmeans_pca = KMeans(n_clusters=3, random_state=42)
kmeans_labels_pca = kmeans_pca.fit_predict(latent_2d_pca)

sil_score_pca = silhouette_score(latent_2d_pca, kmeans_labels_pca)
db_index_pca = davies_bouldin_score(latent_2d_pca, kmeans_labels_pca)
ari_score_pca = adjusted_rand_score(true_labels, kmeans_labels_pca) if 'true_labels' in locals() else None

print("\nPCA-based Clustering Performance:")
print(f"Silhouette Score: {sil_score_pca}")
print(f"Davies-Bouldin Index: {db_index_pca}")
print(f"Adjusted Rand Index: {ari_score_pca if ari_score_pca is not None else 'N/A'}")

Plotting PCA Based

In [ ]:
tsne_pca = TSNE(n_components=2, random_state=42)
latent_2d_pca_tsne = tsne_pca.fit_transform(latent_2d_pca)

# Plot the clusters
plt.figure(figsize=(8, 6))
plt.scatter(latent_2d_pca_tsne[:, 0], latent_2d_pca_tsne[:, 1], c=kmeans_labels_pca, cmap='viridis', marker='o')
plt.title('PCA-based Clustering of Latent Features')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.show()

Plotting VAE based

In [ ]:
# Plot the clusters for VAE
plt.figure(figsize=(8, 6))
plt.scatter(latent_features_df['latent_dim_1'], latent_features_df['latent_dim_2'], c=kmeans_labels, cmap='viridis', marker='o')
plt.title('VAE-based Clustering of Latent Features')
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.show()